# Normal-to-Mixture Bijection Demo

This notebook shows the exact workflow:

1. Draw `z ~ Normal(0, 1)`.
2. Push samples through the bijection `y = t(z)` built by `build_mixture_transform`.
3. Compare transformed samples against direct samples from the target mixture.

In [ ]:
from pathlib import Path
import sys

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from numpyro.distributions import Categorical, MixtureSameFamily, Normal

# Make sure local package imports work whether cwd is repo root or notebooks/.
cwd = Path.cwd()
if (cwd / "src").exists():
    sys.path.insert(0, str(cwd / "src"))
elif (cwd.parent / "src").exists():
    sys.path.insert(0, str(cwd.parent / "src"))

from numpyro_extras.mixture_transform_builder import build_mixture_transform

In [ ]:
# Define a 1D Gaussian mixture.
weights = jnp.array([0.55, 0.45], dtype=jnp.float32)
loc = jnp.array([-1.7, 2.3], dtype=jnp.float32)
scale = jnp.array([0.9, 1.1], dtype=jnp.float32)

mixture = MixtureSameFamily(
    Categorical(probs=weights),
    Normal(loc=loc, scale=scale),
)

build = build_mixture_transform(base="normal", mixture_distribution=mixture)
t = build["transform"]
diagnostics = build["diagnostics"]
diagnostics["approx_summary"]

In [ ]:
# Sample from standard normal, then transform through the bijection.
n = 50_000
key_z, key_mix = jax.random.split(jax.random.PRNGKey(7))

z = jax.random.normal(key_z, shape=(n,), dtype=jnp.float32)
y_from_bijection = t(z)
y_direct_mixture = mixture.sample(key_mix, sample_shape=(n,))

y_from_bijection_np = np.asarray(y_from_bijection)
y_direct_mixture_np = np.asarray(y_direct_mixture)
z_np = np.asarray(z)

quantile_levels = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
q_bij = np.quantile(y_from_bijection_np, quantile_levels)
q_mix = np.quantile(y_direct_mixture_np, quantile_levels)

print("Quantile levels:", quantile_levels)
print("Transformed normal quantiles:", np.round(q_bij, 4))
print("Direct mixture quantiles:", np.round(q_mix, 4))
print("Max abs quantile diff:", float(np.max(np.abs(q_bij - q_mix))))

In [ ]:
# Plot: (left) z -> t(z) mapping, (right) histogram match.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

z_grid = jnp.linspace(-4.5, 4.5, 600, dtype=jnp.float32)
y_grid = np.asarray(t(z_grid))
axes[0].plot(np.asarray(z_grid), y_grid, lw=2.0, color="tab:blue")
axes[0].set_title("Bijection: y = t(z)")
axes[0].set_xlabel("z ~ Normal(0, 1)")
axes[0].set_ylabel("y (mixture space)")
axes[0].grid(alpha=0.25)

bins = 140
x_min = np.percentile(np.concatenate([y_from_bijection_np, y_direct_mixture_np]), 0.2)
x_max = np.percentile(np.concatenate([y_from_bijection_np, y_direct_mixture_np]), 99.8)
axes[1].hist(
    y_direct_mixture_np,
    bins=bins,
    density=True,
    range=(x_min, x_max),
    alpha=0.45,
    label="Direct mixture samples",
    color="tab:orange",
)
axes[1].hist(
    y_from_bijection_np,
    bins=bins,
    density=True,
    range=(x_min, x_max),
    histtype="step",
    linewidth=2.0,
    label="z -> t(z) samples",
    color="tab:blue",
)
axes[1].set_title("Target Distribution Match")
axes[1].set_xlabel("y")
axes[1].set_ylabel("Density")
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.25)

plt.show()

In [ ]:
# Optional sanity check: inverse map recovers the original z values.
z_recovered = np.asarray(t._inverse(y_from_bijection))
roundtrip_err = np.max(np.abs(z_recovered - z_np))
print("Max |t^{-1}(t(z)) - z|:", float(roundtrip_err))